In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
#from sklearn.model_selection import cross_val_score
#from sklearn.metrics import accuracy_score, confusion_matrix,precision_score, recall_score,f1_score,roc_curve, roc_auc_score

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn import svm
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.dummy import DummyClassifier

import mlflow

In [54]:
data = pd.read_csv("../data/processed/transfer_modeling_dataset.csv")
print(data['transfer_success'].value_counts(normalize=True)*100)

transfer_success
0    82.801498
1    17.198502
Name: proportion, dtype: float64


In [56]:
NON_FEATURE_COLUMNS = {
    "transfer_key",
    "player_id",
    "player_full_name",
    "player_name",
    "source_club_id",
    "destination_club_id",
    "source_club_name",
    "destination_club_name",
    "source_club_dataset_name",
    "destination_club_dataset_name",
    "follow_up_window_end",
    "pre_transfer_market_value_date",
    "market_value_180d_prior_date",
    "market_value_365d_prior_date",
    "target_end_market_value_date",
    "target_failure_reason",
    "target_is_eligible",
    "exclusion_reason",
    "date_of_birth",
    "source_country_name",
    "destination_country_name",
    "pre_transfer_market_value_club_id",
    "pre_transfer_market_value_league_id",
    "valuation_cutoff_180d",
    "valuation_cutoff_365d",
    "player_window_start_180d",
    "player_window_end_180d",
    "player_window_start_365d",
    "player_window_end_365d",
    "source_api_cache_file",
    "destination_api_cache_file",
    "player_current_market_value",
    "highest_market_value_in_eur",
    "transfer_season",
    "source_competition_name",
    "destination_competition_name",
    "source_competition_name",
    "destination_competition_name",
}
data.drop(columns=NON_FEATURE_COLUMNS, inplace=True)

In [57]:
# 1. Select the row (e.g., the first row at index 0)
row_data = data.iloc[0]

# 2. Combine values and data types into a single table
result = pd.DataFrame({
    'Value': row_data,
    'DataType': data.dtypes
})

print(result)


                                     Value DataType
transfer_date                   2018-07-01   object
source_competition_id                  IT1   object
destination_competition_id             IT1   object
age_at_transfer                       35.0  float64
position                        Goalkeeper   object
...                                    ...      ...
target_destination_goals_24m           0.0  float64
target_destination_assists_24m         0.0  float64
target_end_market_value           650000.0  float64
target_market_value_delta_24m    -350000.0  float64
transfer_success                         0    int64

[67 rows x 2 columns]


In [58]:
# Returns a list of column names
cols_with_nan = data.columns[data.isna().any()].tolist()
print(cols_with_nan)
print(len(cols_with_nan))

['source_competition_id', 'age_at_transfer', 'foot', 'transfer_fee', 'market_value_in_eur', 'market_value_180d_prior', 'market_value_365d_prior', 'market_value_change_180d', 'market_value_change_365d', 'transfer_fee_to_market_value_ratio', 'transfer_fee_minus_market_value', 'player_goals_per90_180d_pre', 'player_assists_per90_180d_pre', 'player_goals_per90_365d_pre', 'player_assists_per90_365d_pre', 'source_squad_size', 'source_average_age', 'source_foreigners_percentage', 'source_national_team_players', 'source_stadium_seats', 'destination_average_age', 'destination_foreigners_percentage', 'source_win_rate_365d_pre', 'source_points_per_match_365d_pre', 'source_goal_diff_per_match_365d_pre', 'destination_win_rate_365d_pre', 'destination_points_per_match_365d_pre', 'destination_goal_diff_per_match_365d_pre', 'source_api_cached_matches', 'source_api_cached_win_rate', 'source_api_cached_goal_diff_per_match', 'destination_api_cached_matches', 'destination_api_cached_win_rate', 'destination

In [59]:
rows_with_nan = data[data.isna().any(axis=1)]
print(len(rows_with_nan))

6675


In [60]:
# 1. Define your threshold
threshold = 0.2*len(data)  # For example, 20% of the total number of rows

# 2. Identify columns with NaNs > threshold
# data.isna().sum() returns a Series where index is column name and value is NaN count
nan_counts = data.isna().sum()
cols_to_log = nan_counts[nan_counts > threshold]

# 3. Print or Log the results
if not cols_to_log.empty:
    print(f"Columns with more than {threshold} NaNs:")
    print(cols_to_log)
else:
    print("No columns exceed the NaN threshold.")

Columns with more than 1335.0 NaNs:
source_competition_id                         1501
player_goals_per90_180d_pre                   3048
player_assists_per90_180d_pre                 3048
player_goals_per90_365d_pre                   2485
player_assists_per90_365d_pre                 2485
source_squad_size                             1501
source_average_age                            1532
source_foreigners_percentage                  1538
source_national_team_players                  1501
source_stadium_seats                          1501
source_win_rate_365d_pre                      1527
source_points_per_match_365d_pre              1527
source_goal_diff_per_match_365d_pre           1527
source_api_cached_matches                     6675
source_api_cached_win_rate                    6675
source_api_cached_goal_diff_per_match         6675
destination_api_cached_matches                6674
destination_api_cached_win_rate               6674
destination_api_cached_goal_diff_per_match    

In [61]:
data = data.dropna(axis=1,thresh=0.8*len(data))

In [62]:
# Returns a Series with column names and their respective NaN counts
nan_counts = data.isna().sum()

# Filter to show only columns that have more than 0 NaNs
print(nan_counts[nan_counts > 0])

age_at_transfer                               2
foot                                         12
transfer_fee                                559
market_value_in_eur                           5
market_value_180d_prior                     226
market_value_365d_prior                     539
market_value_change_180d                    226
market_value_change_365d                    539
transfer_fee_to_market_value_ratio          559
transfer_fee_minus_market_value             559
destination_average_age                      23
destination_foreigners_percentage            23
destination_win_rate_365d_pre               314
destination_points_per_match_365d_pre       314
destination_goal_diff_per_match_365d_pre    314
dtype: int64


In [63]:
metadata = {
    'features': data.columns.tolist(),
    'data_types': data.dtypes.tolist(),
}
metadata_df = pd.DataFrame(metadata)
metadata_df.to_csv("../data/metadata.csv", index=False)

In [64]:
print(len(data.columns))
rows_with_nan = data[data.isna().any(axis=1)]
print(len(rows_with_nan))
print(len(rows_with_nan)/len(data))

48
1135
0.1700374531835206


In [65]:
# 1. Identify columns with at least one NaN
cols_with_nans = data.columns[data.isna().any()]

# 2. Build a summary table
nan_summary = pd.DataFrame({
    'Column': cols_with_nans,
    'Data Type': data   [cols_with_nans].dtypes.values,
    'NaN Count': data[cols_with_nans].isna().sum().values,
    '% Missing': (data  [cols_with_nans].isna().mean().values * 100).round(2)
})

print("Summary of columns with missing values:")
print(nan_summary.to_string(index=False))

Summary of columns with missing values:
                                  Column Data Type  NaN Count  % Missing
                         age_at_transfer   float64          2       0.03
                                    foot    object         12       0.18
                            transfer_fee   float64        559       8.37
                     market_value_in_eur   float64          5       0.07
                 market_value_180d_prior   float64        226       3.39
                 market_value_365d_prior   float64        539       8.07
                market_value_change_180d   float64        226       3.39
                market_value_change_365d   float64        539       8.07
      transfer_fee_to_market_value_ratio   float64        559       8.37
         transfer_fee_minus_market_value   float64        559       8.37
                 destination_average_age   float64         23       0.34
       destination_foreigners_percentage   float64         23       0.34
           

In [68]:
data['age_at_transfer'].fillna(data['age_at_transfer'].median(), inplace=True)
data['foot'].fillna(data['foot'].mode()[0], inplace=True)
data['transfer_fee'].fillna(data['transfer_fee'].median(), inplace=True)
data['market_value_in_eur'].fillna(data['market_value_in_eur'].median(), inplace=True)
data['market_value_180d_prior'].fillna(data['market_value_180d_prior'].median(), inplace=True)
data['market_value_365d_prior'].fillna(data['market_value_365d_prior'].median(), inplace=True)
data['market_value_change_180d'].fillna(data['market_value_change_180d'].median(), inplace=True)
data['market_value_change_365d'].fillna(data['market_value_change_365d'].median(), inplace=True)
data['transfer_fee_to_market_value_ratio'].fillna(data['transfer_fee_to_market_value_ratio'].median(), inplace=True)
data['transfer_fee_minus_market_value'].fillna(data['transfer_fee_minus_market_value'].median(), inplace=True)
data['destination_average_age'].fillna(data['destination_average_age'].median(), inplace=True)
data['destination_foreigners_percentage'].fillna(data['destination_foreigners_percentage'].median(), inplace=True)
data['destination_win_rate_365d_pre'].fillna(data['destination_win_rate_365d_pre'].median(), inplace=True)
data['destination_points_per_match_365d_pre'].fillna(data['destination_points_per_match_365d_pre'].median(), inplace=True)
data['destination_goal_diff_per_match_365d_pre'].fillna(data['destination_goal_diff_per_match_365d_pre'].median(), inplace=True)

C:\Users\Ali Samy\AppData\Local\Temp\ipykernel_21304\1913183770.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['age_at_transfer'].fillna(data['age_at_transfer'].median(), inplace=True)
C:\Users\Ali Samy\AppData\Local\Temp\ipykernel_21304\1913183770.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['age_at_transfer'].fillna(dat

In [69]:
# 1. Identify columns with at least one NaN
cols_with_nans = data.columns[data.isna().any()]

# 2. Build a summary table
nan_summary = pd.DataFrame({
    'Column': cols_with_nans,
    'Data Type': data   [cols_with_nans].dtypes.values,
    'NaN Count': data[cols_with_nans].isna().sum().values,
    '% Missing': (data  [cols_with_nans].isna().mean().values * 100).round(2)
})

print("Summary of columns with missing values:")
print(nan_summary.to_string(index=False))

Summary of columns with missing values:
Empty DataFrame
Columns: [Column, Data Type, NaN Count, % Missing]
Index: []


In [70]:
train_data, test_data = train_test_split(data, test_size=0.15, shuffle=True, random_state=42)
print(train_data['transfer_success'].value_counts(normalize=True)*100)
print(test_data['transfer_success'].value_counts(normalize=True)*100)
train_data.to_csv("../data/model/train_data.csv", index=False)
test_data.to_csv("../data/model/test_data.csv", index=False) 

transfer_success
0    82.707562
1    17.292438
Name: proportion, dtype: float64
transfer_success
0    83.333333
1    16.666667
Name: proportion, dtype: float64


In [71]:
# I will use simple validation technique just divide train data into train and validation
train_data, val_data = train_test_split(train_data, test_size=0.15, shuffle=True, random_state=42)
print(train_data['transfer_success'].value_counts(normalize=True)*100)
print(val_data['transfer_success'].value_counts(normalize=True)*100)

transfer_success
0    82.496889
1    17.503111
Name: proportion, dtype: float64
transfer_success
0    83.901293
1    16.098707
Name: proportion, dtype: float64


In [72]:
mlflow.set_experiment("Transfer Success Prediction")
mlflow.sklearn.autolog()

y_train =   train_data['transfer_success']
X_train = train_data.drop(columns=['transfer_success'])
y_val =   val_data['transfer_success']
X_val = val_data.drop(columns=['transfer_success'])

# first I will try dummy classifier to set a baseline for the model performance

ZeroR = DummyClassifier(strategy='most_frequent')
ZeroR.fit(X_train, y_train)
y_pred = ZeroR.predict(X_val)
print("ZeroR Classifier")
print(classification_report(y_val, y_pred))

2026/05/02 22:52:53 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'fbc521e7dbe1472890d8b208b342785a', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/05/02 22:52:53 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have mi

ZeroR Classifier
              precision    recall  f1-score   support

           0       0.84      1.00      0.91       714
           1       0.00      0.00      0.00       137

    accuracy                           0.84       851
   macro avg       0.42      0.50      0.46       851
weighted avg       0.70      0.84      0.77       851



c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels wi

In [73]:
# 1. Get the count of each data type
type_counts = train_data.dtypes.value_counts()

print("--- Data Type Counts ---")
print(type_counts)

# 2. See exactly which columns belong to each type
print("\n--- Columns by Type ---")
for dtype in type_counts.index:
    cols = train_data.select_dtypes(include=[dtype]).columns.tolist()
    print(f"{dtype}: {cols}")

--- Data Type Counts ---
float64    32
int64       9
object      7
Name: count, dtype: int64

--- Columns by Type ---
float64: ['age_at_transfer', 'height_in_cm', 'transfer_fee', 'market_value_in_eur', 'pre_transfer_market_value', 'market_value_180d_prior', 'market_value_365d_prior', 'market_value_change_180d', 'market_value_change_365d', 'transfer_fee_to_market_value_ratio', 'transfer_fee_minus_market_value', 'player_matches_180d_pre', 'player_minutes_180d_pre', 'player_goals_180d_pre', 'player_assists_180d_pre', 'player_matches_365d_pre', 'player_minutes_365d_pre', 'player_goals_365d_pre', 'player_assists_365d_pre', 'destination_average_age', 'destination_foreigners_percentage', 'source_matches_365d_pre', 'destination_matches_365d_pre', 'destination_win_rate_365d_pre', 'destination_points_per_match_365d_pre', 'destination_goal_diff_per_match_365d_pre', 'target_destination_matches_24m', 'target_destination_minutes_24m', 'target_destination_goals_24m', 'target_destination_assists_24m',

In [74]:
# now I will handle categorical features with frequency encoding as sklearn does not support categorical features
# directly and I want to avoid one-hot encoding due to high cardinality of some features.
categorical_cols = X_train.select_dtypes(include=['object']).columns
for col in categorical_cols:
    freq_encoding = X_train[col].value_counts() / len(X_train)
    X_train[col] = X_train[col].map(freq_encoding)
    X_val[col] = X_val[col].map(freq_encoding).fillna(0)  # Handle unseen categories in validation set
    test_data[col] = test_data[col].map(freq_encoding).fillna(0)  # Handle unseen categories in test set

test_data.to_csv("../data/model/test_data_encoded.csv", index=False)


In [75]:
# now I will use decision tree classifier
hyperparameters = [5, 10, 15, 20]
for h in hyperparameters:
    params = {
        'criterion': 'entropy',
        'max_depth': 47,
        'min_samples_split': h,
    }
    dt = DecisionTreeClassifier(random_state=42, **params)
    dt.fit(X_train, y_train)
    y_pred = dt.predict(X_val)
    print("Decision Tree Classifier for min_samples_split =", h)
    print(classification_report(y_val, y_pred))

2026/05/02 22:55:24 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '2495f0e046f0454fa4bc22a9dd68fc17', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/05/02 22:55:24 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have mi

Decision Tree Classifier for min_samples_split = 5
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       714
           1       1.00      0.99      1.00       137

    accuracy                           1.00       851
   macro avg       1.00      1.00      1.00       851
weighted avg       1.00      1.00      1.00       851



2026/05/02 22:55:31 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/02 22:55:31 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a war

Decision Tree Classifier for min_samples_split = 10
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       714
           1       1.00      0.99      1.00       137

    accuracy                           1.00       851
   macro avg       1.00      1.00      1.00       851
weighted avg       1.00      1.00      1.00       851



2026/05/02 22:55:38 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/02 22:55:38 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a war

Decision Tree Classifier for min_samples_split = 15
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       714
           1       1.00      0.99      1.00       137

    accuracy                           1.00       851
   macro avg       1.00      1.00      1.00       851
weighted avg       1.00      1.00      1.00       851



2026/05/02 22:55:44 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/02 22:55:44 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a war

Decision Tree Classifier for min_samples_split = 20
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       714
           1       1.00      0.99      1.00       137

    accuracy                           1.00       851
   macro avg       1.00      1.00      1.00       851
weighted avg       1.00      1.00      1.00       851



In [76]:
# now trying logistic regression
hyperparameters = [10,100,1000]
for h in hyperparameters:
    lr = LogisticRegression(max_iter=h)
    lr.fit(X_train,y_train)
    y_pred = lr.predict(X_val)
    print("LogisticRegression for max_iter =", h)
    print(classification_report(y_val, y_pred))

2026/05/02 23:04:58 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '0834a7aee7b7463b81a09a215819b35a', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/05/02 23:04:58 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have mi

LogisticRegression for max_iter = 10
              precision    recall  f1-score   support

           0       0.89      0.83      0.86       714
           1       0.35      0.49      0.41       137

    accuracy                           0.77       851
   macro avg       0.62      0.66      0.63       851
weighted avg       0.81      0.77      0.79       851



2026/05/02 23:05:05 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-

LogisticRegression for max_iter = 100
              precision    recall  f1-score   support

           0       0.89      0.96      0.92       714
           1       0.61      0.37      0.46       137

    accuracy                           0.86       851
   macro avg       0.75      0.66      0.69       851
weighted avg       0.84      0.86      0.85       851



2026/05/02 23:05:12 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-

LogisticRegression for max_iter = 1000
              precision    recall  f1-score   support

           0       0.95      0.97      0.96       714
           1       0.82      0.75      0.78       137

    accuracy                           0.93       851
   macro avg       0.89      0.86      0.87       851
weighted avg       0.93      0.93      0.93       851



In [77]:
y_test = test_data['transfer_success']
X_test = test_data.drop(columns=['transfer_success'])
params = {
    'criterion': 'entropy',
    'max_depth': 47,
    'min_samples_split': 5,
}
dt = DecisionTreeClassifier(random_state=42, **params)
dt.fit(X_train, y_train)
y_pred = dt.predict(X_test)
print(classification_report(y_test,y_pred))

2026/05/02 23:19:20 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '0840ac879882480a9599ac06a9f7d4f7', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/05/02 23:19:20 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have mi

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       835
           1       1.00      1.00      1.00       167

    accuracy                           1.00      1002
   macro avg       1.00      1.00      1.00      1002
weighted avg       1.00      1.00      1.00      1002

